# Эксперимент 16: Kuo-style 2D CNN на MFCC-изображении

**Статья:** Deep Learning-based automated classification of Chinese Speech Sound Disorders (Автоматизированная классификация китайских речевых звуковых нарушений на основе глубокого обучения) 2022

**Ссылка:** [https://arxiv.org/abs/2205.11748](https://arxiv.org/abs/2205.11748)

**Краткое описание модели:** MFCC (и производные) формируются в 2D-представление -> CNN-классификатор.

**Содержание статьи:** Авторы показывают высокую релевантность MFCC-изображений для детекции SSD у детей. Важная идея — свести задачу к устойчивой визуальной классификации time-frequency структуры. Эксперимент переносит этот подход на русские скороговорки.

In [1]:
import sys
from pathlib import Path
import numpy as np
import time
from joblib import Parallel, delayed
import torch
from torch.utils.data import TensorDataset, DataLoader
from torch import nn
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, precision_score, recall_score, classification_report

exp_dir = Path().resolve()
sys.path.insert(0, str(exp_dir.parent))
sys.path.insert(0, str(exp_dir))

from shared import config, data_utils, train_utils
from shared.results_utils import save_result_csv
from model import get_model

In [ ]:
paths_train, paths_val, paths_test, y_train, y_val, y_test, letters_train, letters_val, letters_test = data_utils.get_splits()
n_letters = letters_train.shape[1]
N_MFCC, N_FRAMES = config.N_MFCC, 320

def load_mfcc_img(p):
    return data_utils.extract_mfcc_image(p, n_mfcc=N_MFCC, max_frames=N_FRAMES, stack_delta=True)
X_train = np.stack(Parallel(n_jobs=-1)(delayed(load_mfcc_img)(p) for p in paths_train))
X_val   = np.stack(Parallel(n_jobs=-1)(delayed(load_mfcc_img)(p) for p in paths_val))
X_test  = np.stack(Parallel(n_jobs=-1)(delayed(load_mfcc_img)(p) for p in paths_test))
m, s = X_train.mean(axis=(0,2,3), keepdims=True), X_train.std(axis=(0,2,3), keepdims=True)
s = np.where(s < 1e-6, 1.0, s)
X_train = (X_train - m) / s
X_val   = (X_val - m) / s
X_test  = (X_test - m) / s
print(f"MFCC image: (N, 2, {N_MFCC}, {N_FRAMES})")

MFCC image: (N, 2, 20, 320)


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
train_ds = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(letters_train).float(), torch.from_numpy(y_train).long())
val_ds   = TensorDataset(torch.from_numpy(X_val).float(), torch.from_numpy(letters_val).float(), torch.from_numpy(y_val).long())
test_ds  = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(letters_test).float(), torch.from_numpy(y_test).long())
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

model = get_model(n_mfcc=N_MFCC, n_frames=N_FRAMES, in_channels=2, num_classes=2, n_letters=n_letters, dropout=0.5).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Параметров: {n_params}")

Параметров: 399346


In [4]:
weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = train_utils.get_lr_scheduler(optimizer, patience=config.LR_SCHEDULER_PATIENCE, factor=config.LR_SCHEDULER_FACTOR)
early_stopping = train_utils.EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE)
best_ckpt_path = exp_dir / "best_ckpt.pt"
best_f1 = -1.0

def eval_loader(loader):
    model.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for x, letters, y in loader:
            x, letters, y = x.to(device), letters.to(device), y.to(device)
            pred = model(x, letters).argmax(dim=1)
            all_p.append(pred.cpu().numpy())
            all_t.append(y.cpu().numpy())
    return np.concatenate(all_p), np.concatenate(all_t)

N_EPOCHS = 50
t0 = time.perf_counter()
for epoch in range(N_EPOCHS):
    model.train()
    losses = []
    for x, letters, y in train_loader:
        x, letters, y = x.to(device), letters.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x, letters), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config.DEFAULT_GRAD_CLIP)
        optimizer.step()
        losses.append(loss.item())
    vp, vt = eval_loader(val_loader)
    val_f1 = f1_score(vt, vp, average="macro")
    if val_f1 > best_f1:
        best_f1 = val_f1
        train_utils.save_best_checkpoint(model, best_ckpt_path)
    scheduler.step(val_f1)
    print(f"Epoch {epoch+1}/{N_EPOCHS}  loss={np.mean(losses):.4f}  val_f1_macro={val_f1:.4f}")
    if early_stopping.step(val_f1):
        print(f"Early stopping на эпохе {epoch+1}")
        break
train_time_sec = time.perf_counter() - t0
train_utils.load_best_checkpoint(model, best_ckpt_path, device)
print(f"Обучение: {train_time_sec:.1f} с. Лучший val_f1_macro={best_f1:.4f}")

Epoch 1/50  loss=0.8695  val_f1_macro=0.5292
Epoch 2/50  loss=0.7586  val_f1_macro=0.6582
Epoch 3/50  loss=0.8927  val_f1_macro=0.6764
Epoch 4/50  loss=0.7689  val_f1_macro=0.6198
Epoch 5/50  loss=0.7456  val_f1_macro=0.6812
Epoch 6/50  loss=0.6509  val_f1_macro=0.6617
Epoch 7/50  loss=0.6588  val_f1_macro=0.6758
Epoch 8/50  loss=0.6029  val_f1_macro=0.4983
Epoch 9/50  loss=0.5585  val_f1_macro=0.5535
Epoch 10/50  loss=0.6511  val_f1_macro=0.6405
Epoch 11/50  loss=0.4527  val_f1_macro=0.5880
Epoch 12/50  loss=0.2886  val_f1_macro=0.7295
Epoch 13/50  loss=0.1908  val_f1_macro=0.7251
Epoch 14/50  loss=0.1246  val_f1_macro=0.7021
Epoch 15/50  loss=0.1051  val_f1_macro=0.6068
Epoch 16/50  loss=0.0819  val_f1_macro=0.7122
Epoch 17/50  loss=0.0461  val_f1_macro=0.6820
Epoch 18/50  loss=0.0505  val_f1_macro=0.6752
Epoch 19/50  loss=0.0248  val_f1_macro=0.7111
Epoch 20/50  loss=0.0174  val_f1_macro=0.6792
Epoch 21/50  loss=0.0106  val_f1_macro=0.6943
Epoch 22/50  loss=0.0035  val_f1_macro=0.70

In [ ]:
model.eval()
all_logits = []
with torch.no_grad():
    for x, letters, _ in test_loader:
        x, letters = x.to(device), letters.to(device)
        all_logits.append(model(x, letters).cpu().numpy())
logits = np.concatenate(all_logits)
y_pred = np.argmax(logits, axis=1)
y_proba = torch.softmax(torch.from_numpy(logits), dim=1).numpy()[:, config.CLASS_BAD]

accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
f1_bad = f1_score(y_test, y_pred, pos_label=config.CLASS_BAD)
roc_auc = roc_auc_score(y_test, y_proba)
precision_bad = precision_score(y_test, y_pred, pos_label=config.CLASS_BAD, zero_division=0)
recall_bad = recall_score(y_test, y_pred, pos_label=config.CLASS_BAD, zero_division=0)

print(classification_report(y_test, y_pred, target_names=config.CLASS_NAMES))
print(f"Accuracy: {accuracy:.4f}, F1 macro: {f1_macro:.4f}, ROC-AUC: {roc_auc:.4f}")

save_result_csv(
    exp_dir=exp_dir,
    experiment_id="exp_16_kuo_mfcc_cnn",
    experiment_name="Kuo MFCC 2D CNN",
    model="MFCC+delta 2D CNN",
    accuracy=accuracy,
    f1_macro=f1_macro,
    f1_bad=f1_bad,
    roc_auc=roc_auc,
    precision_bad=precision_bad,
    recall_bad=recall_bad,
    notes="MFCC+delta as 2ch image, 2D CNN",
    num_params=n_params,
    train_time_sec=train_time_sec,
)

              precision    recall  f1-score   support

        good       0.77      0.86      0.81       282
         bad       0.62      0.47      0.53       135

    accuracy                           0.73       417
   macro avg       0.69      0.66      0.67       417
weighted avg       0.72      0.73      0.72       417

Accuracy: 0.7338, F1 macro: 0.6729, ROC-AUC: 0.7635
Результат сохранён в result.csv текущего эксперимента
